# Accentuation Prediction RMT Theory — Validation Demo

This notebook validates the RMT-based theory formulas from `linear_reg_weight_deviation.tex`.

**Key results being validated:**
1. Per-PC ridge regression error: three-term formula
2. Accentuation alignment R_det
3. Accentuation error: `(β*ᵀΣβ*) · (1 - R_det)²`

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import matplotlib.pyplot as plt

from rmt_core import (
    SpectrumKappa,
    ridge_error_per_pc_theory,
    accentuation_error_theory,
    run_monte_carlo,
)

## Setup

In [ ]:
rng = np.random.default_rng(42)

d = 80          # dimension
n = 200         # samples
lam = 0.05      # ridge penalty
sigma = 0.5     # noise std
gamma = d / n

# Power-law spectrum
k = np.arange(1, d+1, dtype=float)
eigenvalues = k**(-1.0)
eigenvalues /= eigenvalues.mean()

# Random orthonormal basis
eigenvectors, _ = np.linalg.qr(rng.standard_normal((d, d)))

# True weight vector
beta_star = rng.standard_normal(d)
beta_proj = eigenvectors.T @ beta_star

# Solve kappa
kappa_fn = SpectrumKappa(eigenvalues, gamma)
kappa = kappa_fn(lam)
print(f'κ({lam}) = {kappa:.4f}')

## Per-PC error: Theory vs Monte Carlo

In [ ]:
err_theory, t1, t2, t3 = ridge_error_per_pc_theory(
    eigenvalues, beta_proj, kappa, sigma, n)

mc = run_monte_carlo(n, eigenvalues, eigenvectors, beta_star, sigma, lam,
                     n_trials=2000, rng=rng)
err_sim = mc['error_per_pc']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.semilogy(err_sim, 'o', ms=3, alpha=0.6, label='MC')
ax.semilogy(err_theory, '-', lw=1.5, label='Theory total')
ax.semilogy(t1, '--', lw=1, label='Term1: overshrinkage')
ax.semilogy(t2, '--', lw=1, label='Term2: finite-n signal')
ax.semilogy(t3, '--', lw=1, label='Term3: finite-n noise')
ax.set_xlabel('PC index k'); ax.set_ylabel('E[(uₖᵀΔβ)²]')
ax.set_title('Per-PC error'); ax.legend(fontsize=7)

ax = axes[1]
ax.loglog(err_theory, err_sim, 'o', ms=3, alpha=0.5)
lims = [min(err_theory.min(), err_sim.min()), max(err_theory.max(), err_sim.max())]
ax.plot(lims, lims, 'k--', lw=1)
ax.set_xlabel('RMT theory'); ax.set_ylabel('MC')
ax.set_title('Theory vs MC')
plt.tight_layout()

## Accentuation error vs sample size n

In [ ]:
n_values = [60, 80, 100, 150, 200, 300, 500]
theory_errors, mc_errors, theory_R, mc_R = [], [], [], []

for n_val in n_values:
    gam = d / n_val
    kfn = SpectrumKappa(eigenvalues, gam)
    kap = kfn(lam)
    err_th, R_th, _ = accentuation_error_theory(eigenvalues, beta_proj, kap, sigma, n_val)
    mc_v = run_monte_carlo(n_val, eigenvalues, eigenvectors, beta_star, sigma, lam,
                           n_trials=1000, rng=rng)
    theory_errors.append(err_th); mc_errors.append(mc_v['acc_error'])
    theory_R.append(R_th); mc_R.append(mc_v['acc_alignment'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(n_values, theory_R, '-o', label='Theory R_det')
axes[0].plot(n_values, mc_R, '--s', label='MC R')
axes[0].set_xlabel('n'); axes[0].set_ylabel('Alignment R'); axes[0].legend()
axes[0].set_title('Accentuation alignment vs n')

axes[1].plot(n_values, theory_errors, '-o', label='Theory')
axes[1].plot(n_values, mc_errors, '--s', label='MC')
axes[1].set_xlabel('n'); axes[1].set_ylabel('Accentuation error'); axes[1].legend()
axes[1].set_title('Accentuation error vs n')
plt.tight_layout()